In [1]:
import nltk
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.datasets import DenoisingAutoEncoderDataset
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer import model, SentenceTransformer, losses

nltk.download('punkt_tab')  # 下载 punkt_tab 分词器

mnli = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(25_000))
flat_sentences = list(mnli["premise"]) + list(mnli["hypothesis"]) # 共 50,000 条句子

# # 为输入数据添加噪声
damaged_data = DenoisingAutoEncoderDataset(flat_sentences)

train_dataset = {"damaged_sentence": [], "original_sentence": []}
for data in damaged_data:  # 这里会用到 `punkt_tab` 分词器
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])
train_dataset = Dataset.from_dict(train_dataset)

print(train_dataset[3])

print(train_dataset["damaged_sentence"][0])

# 定义评估器，使用语义文本相似度基准(Semantic Textual Similarity Benchmark, STSB)
# 这是一个由人工标注的句子对数据集，相似度分数在 1 ~ 5 之间
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]], # 值转换为 0~1 之间
    main_similarity="cosine"
)

word_embedding_model = model.Transformer("bert-base-uncased")
pooling_model = model.Pooling(word_embedding_model.get_embedding_dimension(), "cls")
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model], device="cuda")

# 专门的损失函数
# transformers 5.0.0 之后，tie_encoder_decoder 必须为 False, 并设置  decoder_name_or_path
# transformers 当前版本 5.8.0
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model,
    decoder_name_or_path="bert-base-uncased",
    tie_encoder_decoder=False
)
train_loss.decoder = train_loss.decoder.to("cuda")

# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()
embedding_model.save("tsdae_embedding_model")

[nltk_data] Downloading package punkt_tab to /home/yanbin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


{'damaged_sentence': 'you All this is their', 'original_sentence': 'How do you know? All this is their information again.'}
cream skimming two - and.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
import torch
torch.save(train_loss.decoder.state_dict(), "tsdae_embedding_model/decoder.pt")

In [56]:
train_dataset[3]

{'damaged_sentence': 'do you know this is information',
 'original_sentence': 'How do you know? All this is their information again.'}

In [55]:
evaluator(embedding_model)

{'pearson_cosine': 0.7186646635148979, 'spearman_cosine': 0.7280383796512555}